In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import subprocess
import concurrent.futures
from joblib import Parallel, delayed

In [2]:
# latencies: 50, 90 150, 210
default_region = ['us-central1-c']
# regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

regions = ['us-central1-c', 'us-central1-c', 'us-central1-c', 'us-central1-c']


# Regions

# num_nodes = 4
zone_no = 0
n_clients = 1
for num_nodes in  [8]:
# for zone_no in  [0,1,2,3, 4]:


    project = "research-488322"
    zone = "us-central1-c"
    machine_type = "e2-standard-2"
    image_family = "tsm-sc-family"  # your custom image
    subnet = "default"
    gcp_username = "tejas"

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")

    

    # Create commands list
    commands = []
    
    for i in range(num_nodes):

        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        cmd = f'''
        gcloud compute instances create tsm-sc-{i:03} \
            --project={project} \
            --zone={zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())


    for i in range(n_clients):

        if i < int(n_clients/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        cmd = f'''
        gcloud compute instances create tsm-sc-{i+num_nodes:03} \
            --project={project} \
            --zone={zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
    
    
    def run_command(command):
        print(f"Running: {command}")
        return subprocess.call(command, shell=True)
    
    
    # #Parallel instance creation
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=48) as executor:
        futures = [executor.submit(run_command, cmd) for cmd in commands]
        concurrent.futures.wait(futures)
    
    print("All instances launched.")


    # Wait a bit for IPs to propagate
    import time
    # time.sleep(30)
    

    # Get IPs
    os.system('gcloud compute instances list --filter="name~\'tsm-sc-\'" '
              '--format="value(networkInterfaces[0].networkIP)" > tsm_ips.txt')
    os.system("sed -i '$d' tsm_ips.txt")

    with open('tsm_ips.txt', 'r') as f:
        iplist = [line.strip() for line in f.readlines()]
    
    print("🎯 Instance IPs:", iplist)
    node1_ip = iplist[0]
    print(f"Client will connect to node1 at: {node1_ip}")


    os.system('git add .; git commit -m "testing"; git push')
   

    n_collection = 100
    os.system('make -j8')
    
    

    def kill_stellar_private(i):


        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        
        remote_command = f"""\
    cd /home/tejas/stellar-private; \
    sudo pkill -9 stellar-core; \
    """
        
        # Construct the full gcloud command
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
        
        print(f"Executing: {command}")
    
        output = os.system(command)
        print(f"Return code for tsm-sc-{i:03}: {output}")
    
    
    results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    

    
    
    
    def git_pull_stellar(i):
        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core; \
    git pull"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=48)(delayed(git_pull_stellar)(i) for i in range(len(iplist)))
    print(results)
    

    import shutil
    
    if os.path.exists('../stellar-private'):
        
        shutil.rmtree('../stellar-private')
    os.mkdir('../stellar-private')
    
    
    os.system('cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh')
    
    os.system('cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; ./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh')
    
    # --- Configuration ---
    line_to_add = "SEND_CUSTOM_MESSAGE=true"
    target_file = "../stellar-private/node1/stellar-core.cfg" 
    
    # --- The os.system() Command ---
    # This command prepends the line to the target_file on your local machine.
    os.system(f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}')

    target_file = "../stellar-private/node2/stellar-core.cfg" 
    line_to_add = "MEMORY_PROF=true"

    os.system(f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}')
    
    print(f"The line '{line_to_add}' has been prepended to {target_file}.")
    

    def compile_stellar(i):

        if i < int(num_nodes/2):
            zone = default_region[0]
        else:
            zone = regions[zone_no]
        command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
    cd stellar-core;g++ -O2 -std=c++17 -pthread \
    -I/home/tejas/stellar-core/src \
    /home/tejas/stellar-core/shab_client.cpp \
    -o /home/tejas/stellar-core/shab_client; make -j16; cd; sudo rm -r stellar-private"'
        print(command)
        output = os.system(command)
        print(output)
    
    # Execute in parallel like your example
    results = Parallel(n_jobs=48)(delayed(compile_stellar)(i) for i in range(len(iplist)))
    print(results)



    for sleep_time in [0, 100, 200, 500, 1000, 2000]:

        
    
    
        def clean_stellar_private(i):
    
            if i < int(num_nodes/2):
                zone = default_region[0]
            else:
                zone = regions[zone_no]
    
            
            remote_command = f"""\
        cd /home/tejas; \
        sudo rm -r stellar-private; \
        """
            
            # Construct the full gcloud command
            command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
            
            print(f"Executing: {command}")
        
            output = os.system(command)
            print(f"Return code for tsm-sc-{i:03}: {output}")
        
        
        results = Parallel(n_jobs=20)(delayed(clean_stellar_private)(i) for i in range(num_nodes))
        
    
        def copy_folder_to_instance(i, source_folder = '/home/tejas/stellar-private',destination_path = '/home/tejas/stellar-private' ):
            """
            Constructs and executes the gcloud compute scp command to copy a folder
            to a specific GCP instance.
            """
    
    
            if i < int(num_nodes/2):
                zone = default_region[0]
            else:
                zone = regions[zone_no]
            
            instance_name = f"tsm-sc-{i:03}"
            
            # The --recurse flag is crucial for copying folders
            # The format is: gcloud compute scp --recurse [LOCAL_SRC] [USER]@[INSTANCE_NAME]:[REMOTE_DEST]
            command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
        --recurse "{source_folder}" "{instance_name}:{destination_path}"'
        
            print(f"Executing command for {instance_name}: {command}")
            
            # os.system executes the command and returns the exit status (0 for success)
            output = os.system(command)
            
            print(f"Command for {instance_name} finished with exit code: {output}")
            
            return (instance_name, output)
        
        
        results = Parallel(n_jobs=48)(
            delayed(copy_folder_to_instance)(i) for i in range(num_nodes)
        )
        
        
    
    
    
        
        def run_stellar_private(i):
    
    
            if i < int(num_nodes/2):
                zone = default_region[0]
            else:
                zone = regions[zone_no]
            # Calculate the node number (assuming i starts at 0, node starts at 1)
            node_number = i + 1 
            instance_name = f"tsm-sc-{i:03}"
            
            # ----------------------------------------------------------------------------------
            # FIX: Use nohup, redirect I/O to a log file, and add ' & disown'
            # '2>&1' redirects stderr to stdout. '> log.txt' redirects stdout to a file.
            # '< /dev/null' ensures the process doesn't wait for input.
            # ----------------------------------------------------------------------------------
            remote_command = f"""\
        cd /home/tejas/stellar-private; \
        nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
        > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
        """
            
            # Construct the full gcloud command
            command = f'gcloud compute ssh --zone "{zone}" "{instance_name}" --project "{project}" --command "{remote_command}"'
            
            print(f"Executing: {command}")
            
            # os.system should now return immediately because the remote shell exits
            output = os.system(command)
            print(f"Return code for {instance_name}: {output}")
    
    
    
    
        
            
        results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
    
    
    
    
        
        def run_stellar_client(i):
    
    
            if i < int(num_nodes/2):
                zone = default_region[0]
            else:
                zone = regions[zone_no]
            # Calculate the node number (assuming i starts at 0, node starts at 1)
            node_number = i + 1 
            instance_name = f"tsm-sc-{i:03}"
            
            # ----------------------------------------------------------------------------------
            # FIX: Use nohup, redirect I/O to a log file, and add ' & disown'
            # '2>&1' redirects stderr to stdout. '> log.txt' redirects stdout to a file.
            # '< /dev/null' ensures the process doesn't wait for input.
            # ----------------------------------------------------------------------------------
            remote_command = f"""\
        cd /home/tejas/stellar-private; \
        nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 400 3600000 100 0 {sleep_time}\
        > stellar-client.log 2>&1 < /dev/null & disown
        """
            
            # Construct the full gcloud command
            command = f'gcloud compute ssh --zone "{zone}" "{instance_name}" --project "{project}" --command "{remote_command}"'
            
            print(f"Executing: {command}")
            
            # os.system should now return immediately because the remote shell exits
            output = os.system(command)
            print(f"Return code for {instance_name}: {output}")
    
        # Corrected Loop (to run 0, 1, 2, 3)
        results = Parallel(n_jobs=48)(delayed(run_stellar_private)(i) for i in range(num_nodes))
        
    
        # time.sleep(3)
        # for i in range(num_nodes):
        
            # run_stellar_private(num_nodes-i-1)
            # time.sleep(2)
        # run_stellar_private(0)
        print(results)
        print("All SSH commands executed. Nodes should be starting up in the background.")
        
        time.sleep(60)
        results = Parallel(n_jobs=48)(delayed(run_stellar_client)(i) for i in ([num_nodes]))
    
        time.sleep(210)
        
        # results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in [1])
        # time.sleep(10)
        # results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in [2])
        # time.sleep(10)
        # results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in [3])
        # time.sleep(10)
        # results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in [4])
        # time.sleep(10)
        # results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in [5])
        # time.sleep(10)
    
        # time.sleep(50)
        
        
        results = Parallel(n_jobs=48)(delayed(kill_stellar_private)(i) for i in range(num_nodes))
        
        
    
        remote_base_folder = "/home/tejas/stellar-private" # The base path on the GCP instance
        # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "collection_"+str(n_collection)+"_rounds_" + str(num_nodes) + "_v2" 
        # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "memory_"+ str(num_nodes) + "_no_cleanup_v2" 
        local_base_destination = "/home/tejas/work/experiments/shabdiz/" + "shab_sleepbtw_request_"+ str(num_nodes) + '_' + str(sleep_time) 
    
        # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "collection_"+str(n_collection)+"_rounds_" + str(num_nodes) + "_zone_" + str(zone_no)+"_v2" 
        # local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "memory_" + str(num_nodes) + "_node_failure"
        
        # Ensure the local base destination directory exists
        os.makedirs(local_base_destination, exist_ok=True)
        
        
        def copy_folder_from_instance(i):
    
            if i < int(num_nodes/2):
                zone = default_region[0]
            else:
                zone = regions[zone_no]
                
            """
            Constructs and executes the gcloud compute scp command to copy a specific 
            nodeN folder from instance i to a local folder named after the instance.
            """
            instance_name = f"tsm-sc-{i:03}"
            
            # Calculate the node number (assuming i starts at 0, node starts at 1)
            node_number = i + 1 
            node_folder = f"node{node_number}"
        
            # 1. Define the specific REMOTE source path on the instance
            # Example: /home/tejas/stellar-private/node1
            remote_source_path = os.path.join(remote_base_folder, node_folder)
            
            # 2. Define the LOCAL destination path
            # We'll use the instance name for the subfolder to keep backups separate
            local_destination_path = os.path.join(local_base_destination, instance_name)
            os.makedirs(local_destination_path, exist_ok=True)
            
            # The SCp command requires the remote path to be formatted as:
            # [INSTANCE_NAME]:[REMOTE_SRC]
            remote_source = f"{instance_name}:{remote_source_path}"
            
            # The command reverses the source (remote) and destination (local)
            command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
        --recurse "{remote_source}" "{local_destination_path}"'
        
            print(f"Executing command to copy {node_folder} from {instance_name}: {command}")
            
            # os.system executes the command and returns the exit status (0 for success)
            output = os.system(command)
            
            print(f"Copy from {instance_name} finished with exit code: {output}")
            
            return (instance_name, output)
    
    
    
        # ---
        # Execute the copy operation in parallel
        # ---
        
        results = Parallel(n_jobs=48)(
            delayed(copy_folder_from_instance)(i) for i in range(3)
        )
    
        def copy_client_log():
            """
            Copies shab_client.log from the last instance (client node) to local destination.
            """
            i = num_nodes
            instance_name = f"tsm-sc-{i:03}"
            
            if i < int(num_nodes/2):
                zone = default_region[0]
            else:
                zone = regions[zone_no]
            
            remote_source = f"{instance_name}:/home/tejas/stellar-private/stellar-client.log"
            local_destination_path = local_base_destination
            os.makedirs(local_destination_path, exist_ok=True)
            
            command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
        "{remote_source}" "{local_destination_path}/client.log"'
            
            print(f"Copying shab_client.log from {instance_name}...")
            output = os.system(command)
            print(f"Copy finished with exit code: {output}")
            
            return (instance_name, output)
    
        
        copy_client_log()
        
        print("\n--- Summary of Download Results ---")
        print(results)
        

    # # Fetch all tsm-sc-* instances across ALL zones
    # fetch_cmd = f'''
    # gcloud compute instances list \
    #     --project={project} \
    #     --filter="name~'^tsm-sc-'" \
    #     --format="value(name,zone)"
    # '''
    
    # output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    # instances = []
    
    # for line in output.splitlines():
    #     if line.strip():
    #         name, zone = line.split()
    #         instances.append((name, zone))
    
    # print("\n➡ Existing instances to delete:")
    # for name, zone in instances:
    #     print(f"  - {name} ({zone})")
    
    # def delete_instance(name, zone):
    #     cmd = f'''
    #     gcloud compute instances delete {name} \
    #         --zone={zone} \
    #         --project={project} \
    #         --quiet
    #     '''
    #     print(f"🗑️ Deleting {name} in {zone}")
    #     return subprocess.call(cmd, shell=True)
    
    # if instances:
    #     with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
    #         futures = [
    #             executor.submit(delete_instance, name, zone)
    #             for name, zone in instances
    #         ]
    #         concurrent.futures.wait(futures)
    
    #     print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    # else:
    #     print("\n✔ No tsm-sc-* instances found.\n")


➡ Existing instances to delete:

✔ No tsm-sc-* instances found.

Running: gcloud compute instances create tsm-sc-000             --project=research-488322             --zone=us-central1-c             --machine-type=e2-standard-2             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default             --can-ip-forward             --maintenance-policy=MIGRATE             --provisioning-model=STANDARD             --service-account=254510644191-compute@developer.gserviceaccount.com             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append             --tags=http-server,https-server             --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced             

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-c/instances/tsm-sc-000].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-000  us-central1-c  e2-standard-2               10.128.0.34  34.68.183.121  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-c/instances/tsm-sc-007].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-c/instances/tsm-sc-006].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert

 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-007  us-central1-c  e2-standard-2               10.128.0.37  35.188.192.100  RUNNING
NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-006  us-central1-c  e2-standard-2               10.128.0.58  35.184.72.144  RUNNING


...

Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-004" --project "research-488322" --command "        cd /home/tejas;         sudo rm -r stellar-private;         "
Return code for tsm-sc-004: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-003" --project "research-488322" --command "        cd /home/tejas/stellar-private;         nohup /home/tejas/stellar-core/src/stellar-core run --conf node4/stellar-core.cfg         > node4/stellar-core.log 2>&1 < /dev/null & disown
        "
Return code for tsm-sc-003: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-006" --project "research-488322" --command "        cd /home/tejas;         sudo rm -r stellar-private;         "
Return code for tsm-sc-006: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-000" --project "research-488322" --command "        cd /home/tejas/stellar-private;         nohup /home/tejas/stellar-core/src/stellar-core run --conf node1/stellar-core.cfg         > node1/stell

Exception ignored in: <function ResourceTracker.__del__ at 0x74e14677e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x721e64d8e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

[None, None, None, None, None, None, None, None]
All SSH commands executed. Nodes should be starting up in the background.
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-004" --project "research-488322" --command "        cd /home/tejas;         sudo rm -r stellar-private;         "
Return code for tsm-sc-004: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-003" --project "research-488322" --command "        cd /home/tejas/stellar-private;         nohup /home/tejas/stellar-core/src/stellar-core run --conf node4/stellar-core.cfg         > node4/stellar-core.log 2>&1 < /dev/null & disown
        "
Return code for tsm-sc-003: 0
Executing command for tsm-sc-000: gcloud compute scp --zone "us-central1-c" --project "research-488322"         --recurse "/home/tejas/stellar-private" "tsm-sc-000:/home/tejas/stellar-private"
Command for tsm-sc-000 finished with exit code: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-008" --project "research-488322" -

Exception ignored in: <function ResourceTracker.__del__ at 0x71076d58e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x79c87b58a020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

KeyboardInterrupt: 

[None, None, None, None, None, None, None, None]
All SSH commands executed. Nodes should be starting up in the background.


Exception ignored in: <function ResourceTracker.__del__ at 0x77434b98e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x72b5fe18e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

Copying shab_client.log from tsm-sc-008...
Copy finished with exit code: 0

--- Summary of Download Results ---
[('tsm-sc-000', 0), ('tsm-sc-001', 0), ('tsm-sc-002', 0)]
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-006" --project "research-488322" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return code for tsm-sc-006: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-004" --project "research-488322" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-004: 256
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-007" --project "research-488322" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return code for tsm-sc-007: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-005" --project "research-488322" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-005: 256
Executing: gcloud comp

Exception ignored in: <function ResourceTracker.__del__ at 0x79bc5798a020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x730301796020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-000" --project "research-488322" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return code for tsm-sc-000: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-001" --project "research-488322" --command "    cd /home/tejas/stellar-private;     nohup /home/tejas/stellar-core/src/stellar-core run --conf node2/stellar-core.cfg     > node2/stellar-core.log 2>&1 < /dev/null & disown
    "
Return code for tsm-sc-001: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-008" --project "research-488322" --command "    cd /home/tejas/stellar-private;     nohup /home/tejas/stellar-core/shab_client 10.128.0.34 12000 400 8000000 100 0 0    > stellar-client.log 2>&1 < /dev/null & disown
    "
Return code for tsm-sc-008: 0


...

Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-004" --project "research-488322" --command "    cd /home/tejas/stellar-private;     nohup /home/tejas/stellar-core/src/stellar-core run --conf node5/stellar-core.cfg     > node5/stellar-core.log 2>&1 < /dev/null & disown
    "
Return code for tsm-sc-004: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-006" --project "research-488322" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return code for tsm-sc-006: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-003" --project "research-488322" --command "    cd /home/tejas/stellar-private;     nohup /home/tejas/stellar-core/src/stellar-core run --conf node4/stellar-core.cfg     > node4/stellar-core.log 2>&1 < /dev/null & disown
    "
Return code for tsm-sc-003: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-008" --project "research-488322" --command "    cd /home/tejas/stellar-private;     nohup /home/tejas/stellar-

Exception ignored in: <function ResourceTracker.__del__ at 0x7c06ea38e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7469f6d96020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

[None, None, None, None, None, None, None, None]
All SSH commands executed. Nodes should be starting up in the background.
Executing command for tsm-sc-000: gcloud compute scp --zone "us-central1-c" --project "research-488322"     --recurse "/home/tejas/stellar-private" "tsm-sc-000:/home/tejas/stellar-private"
Command for tsm-sc-000 finished with exit code: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-000" --project "research-488322" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-000: 256
Executing command for tsm-sc-003: gcloud compute scp --zone "us-central1-c" --project "research-488322"     --recurse "/home/tejas/stellar-private" "tsm-sc-003:/home/tejas/stellar-private"
Command for tsm-sc-003 finished with exit code: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-000" --project "research-488322" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return co

Exception ignored in: <function ResourceTracker.__del__ at 0x79c71778e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7a2cc2f8e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

Copying shab_client.log from tsm-sc-008...
Copy finished with exit code: 0

--- Summary of Download Results ---
[('tsm-sc-000', 0), ('tsm-sc-001', 0), ('tsm-sc-002', 0)]


In [4]:

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


➡ Existing instances to delete:
  - tsm-sc-000 (us-central1-c)
  - tsm-sc-001 (us-central1-c)
  - tsm-sc-002 (us-central1-c)
  - tsm-sc-003 (us-central1-c)
  - tsm-sc-004 (us-central1-c)
  - tsm-sc-005 (us-central1-c)
  - tsm-sc-006 (us-central1-c)
  - tsm-sc-007 (us-central1-c)
  - tsm-sc-008 (us-central1-c)
🗑️ Deleting tsm-sc-000 in us-central1-c
🗑️ Deleting tsm-sc-001 in us-central1-c
🗑️ Deleting tsm-sc-002 in us-central1-c
🗑️ Deleting tsm-sc-003 in us-central1-c
🗑️ Deleting tsm-sc-004 in us-central1-c
🗑️ Deleting tsm-sc-005 in us-central1-c
🗑️ Deleting tsm-sc-006 in us-central1-c
🗑️ Deleting tsm-sc-007 in us-central1-c
🗑️ Deleting tsm-sc-008 in us-central1-c
Executing command for tsm-sc-004: gcloud compute scp --zone "us-central1-c" --project "research-488322"     --recurse "/home/tejas/stellar-private" "tsm-sc-004:/home/tejas/stellar-private"
Command for tsm-sc-004 finished with exit code: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-006" --project "research-488

Exception ignored in: <function ResourceTracker.__del__ at 0x764ae2786020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-005" --project "research-488322" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return code for tsm-sc-005: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-007" --project "research-488322" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-007: 256
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-000" --project "research-488322" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return code for tsm-sc-000: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-005" --project "research-488322" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-005: 256
Executing command to copy node2 from tsm-sc-001: gcloud compute scp --zone "us-central1-c" --project "research-488322"     --recurse "tsm-sc-001:/home/tejas/stellar-private/node2" "/home/tejas

Exception ignored in: <function ResourceTracker.__del__ at 0x7149dc38e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7d7bcc986020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-004" --project "research-488322" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return code for tsm-sc-004: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-001" --project "research-488322" --command "    cd /home/tejas/stellar-private;     nohup /home/tejas/stellar-core/src/stellar-core run --conf node2/stellar-core.cfg     > node2/stellar-core.log 2>&1 < /dev/null & disown
    "
Return code for tsm-sc-001: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-005" --project "research-488322" --command "    cd /home/tejas/stellar-private;     nohup /home/tejas/stellar-core/src/stellar-core run --conf node6/stellar-core.cfg     > node6/stellar-core.log 2>&1 < /dev/null & disown
    "
Return code for tsm-sc-005: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-007" --project "research-488322" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return c

...

Exception ignored in: <function ResourceTracker.__del__ at 0x772546d86020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-001" --project "research-488322" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return code for tsm-sc-001: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-005" --project "research-488322" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-005: 256
Executing command to copy node2 from tsm-sc-001: gcloud compute scp --zone "us-central1-c" --project "research-488322"     --recurse "tsm-sc-001:/home/tejas/stellar-private/node2" "/home/tejas/work/experiments/shabdiz/shab_sleepbtw_request_8_1000/tsm-sc-001"
Copy from tsm-sc-001 finished with exit code: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-006" --project "research-488322" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-006: 0
Executing command for tsm-sc-007: gcloud compute scp --zone "us-central1-c" --pr

Exception ignored in: <function ResourceTracker.__del__ at 0x7ce05358e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x79e456d92020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-002" --project "research-488322" --command "    cd /home/tejas;     sudo rm -r stellar-private;     "
Return code for tsm-sc-002: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-006" --project "research-488322" --command "    cd /home/tejas/stellar-private;     sudo pkill -9 stellar-core;     "
Return code for tsm-sc-006: 256
Executing command to copy node3 from tsm-sc-002: gcloud compute scp --zone "us-central1-c" --project "research-488322"     --recurse "tsm-sc-002:/home/tejas/stellar-private/node3" "/home/tejas/work/experiments/shabdiz/shab_sleepbtw_request_8_1000/tsm-sc-002"
Copy from tsm-sc-002 finished with exit code: 0


Exception ignored in: <function ResourceTracker.__del__ at 0x74fb1d18a020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
